<a href="https://colab.research.google.com/github/Shayenvi15/LLM-AI-Projects/blob/main/PDF_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 📄 Chat with your PDF - LangChain + Groq + Gradio UI

In [ ]:
!pip install langchain-openai langchain-community huggingface_hub PyPDF2 langchain-huggingface faiss-cpu langchain-groq gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.1 MB/s eta 0:00:00


In [ ]:
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

import os
import gradio as gr
import tempfile


In [ ]:
conversation = None

In [ ]:
def get_pdf_text(pdf_path):
    text = ""
    pdf_reader = PdfReader(pdf_path)
    for page in pdf_reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text
    return text


In [ ]:
def get_text_chunks(raw_text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    return text_splitter.split_text(raw_text)


In [ ]:
def get_vectorstore(text_chunks):
    embeddings = HuggingFaceEmbeddings(model_name="hkunlp/instructor-xl")
    vectorstore = FAISS.from_texts(texts=text_chunks, embedding=embeddings)
    return vectorstore


In [ ]:
from google.colab import userdata

def get_conversation_chain(vectorstore):
    llm = ChatGroq(
        model_name="llama-3-70b-8192",
        temperature=0.7,
        request_timeout=30,
        api_key=userdata.get("GROQ_API_KEY")
    )
    memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(),
        memory=memory
    )
    return chain

In [ ]:
def process_pdf(file):
    try:
        if file is None:
            return "No file uploaded."

        loader = PyPDFLoader(file.name)
        pages = loader.load_and_split()

        instructor_embeddings = HuggingFaceInstructEmbeddings(model_name="hkunlp/instructor-xl")
        vectorstore = FAISS.from_documents(pages, instructor_embeddings)

        retriever = vectorstore.as_retriever()

        qa_chain = RetrievalQA.from_chain_type(
            llm=ChatGroq(temperature=0, model_name="llama3-8b-8192", groq_api_key=userdata.get("GROQ_API_KEY")),
            chain_type="stuff",
            retriever=retriever
        )

        global chain
        chain = qa_chain  # Save it globally for use during chat

        return "PDF processed successfully ✅"

    except Exception as e:
        import traceback
        traceback_str = traceback.format_exc()
        print("Error while processing PDF:\n", traceback_str)
        return f"Error: {str(e)}"

In [ ]:
def chat_with_pdf(user_query):
    global conversation
    if conversation is None:
        return "❌ Please upload a PDF first."

    response = conversation({'question': user_query})
    return response['answer']


In [ ]:
with gr.Blocks(theme=gr.themes.Base(primary_hue="green", secondary_hue="green")) as demo:
    gr.Markdown("## 📄💬 Chat with your PDF - RAG powered by LangChain + Groq", elem_classes="dark")

    with gr.Row():
        pdf_file = gr.File(label="Upload PDF", file_types=[".pdf"])
        upload_button = gr.Button("Process PDF")

    status_output = gr.Textbox(label="Status")

    with gr.Row():
        user_query = gr.Textbox(label="Ask a question")
        send_button = gr.Button("Ask")

    chatbot_output = gr.Textbox(label="Response")

    upload_button.click(fn=process_file, inputs=[pdf_file], outputs=[status_output])
    send_button.click(fn=chat_with_pdf, inputs=[user_query], outputs=[chatbot_output])

demo.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2e80d714e243754540.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
